# 🔴 Layoff Pulse 2026
### Reading and Forecasting Real-World Signal from the 2026 Tech Layoff Wave
**DS Club Presentation — Live pipeline, not a cleaned Kaggle notebook**

Everything below hits **live sources** at run time. Nothing is a static CSV.
That's the point: real data science is messy, ambiguous, and has to be
scraped and interrogated before any conclusion is trustworthy.

**Pipeline:** scrape → clean → explore → question → forecast → question again


## 0. Setup — pull the pipeline modules into this Colab session

In [ ]:
# Clone the repo so we're importing the same .py modules used in development,
# not pasting code into notebook cells (the .py files ARE the pipeline;
# this notebook orchestrates them).
!git clone https://github.com/YOUR-ORG/layoff-pulse-2026.git
%cd layoff-pulse-2026

!pip install -q requests beautifulsoup4 feedparser rapidfuzz pandas plotly statsmodels ipywidgets lxml


In [ ]:
import sys, os
sys.path.append('scraper')
sys.path.append('pipeline')

import pandas as pd
import tracker_scraper as ts
import news_scraper as ns
import clean as cl
import eda
import forecast as fc

import ipywidgets as widgets
from IPython.display import display


## 1. Hook — live headline number
First thing the audience sees: a real number, pulled live, before any code is explained.


In [ ]:
hook = ts.scrape_trueup_headline()
print("LIVE right now, numbers found on trueup.io/layoffs:")
print(hook["raw_numbers_found"])
print()
print(hook["raw_text_snippet"][:300])


## 2. Scrape — structured tracker + news, live

Watch this run live: it tries the layoffs.fyi Airtable view first, falls back
to Apify, falls back to WARN Act filings. **Whatever happens on stage is real** —
if a source fails, that failure IS the lesson about scraper fragility.

> Set these env vars beforehand if you have them (optional — the pipeline
> degrades gracefully without them):
> `LAYOFFSFYI_AIRTABLE_BASE`, `AIRTABLE_PAT`, `APIFY_TOKEN`, `WARN_STATE_URL`


In [ ]:
raw_tracker_df = ts.get_live_tracker_data(
    airtable_base_id=os.environ.get("LAYOFFSFYI_AIRTABLE_BASE"),
    warn_state_url=os.environ.get("WARN_STATE_URL"),
)
print(f"\nRaw rows pulled: {len(raw_tracker_df)}")
raw_tracker_df.head(10)


In [ ]:
# Live news scrape — RSS feeds filtered to layoff-relevant headlines
headlines_df = ns.fetch_all_feeds()
print(f"Layoff-relevant headlines found: {len(headlines_df)}")
headlines_df.head(10)


> 🛑 **Optional live "slow path" demo**: Selenium against layoffs.fyi's
> rendered page directly, to *show* why we didn't build the pipeline this way.
> Skip this cell if short on time.


In [ ]:
# OPTIONAL — slow, shown for contrast only
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# import time
#
# driver = webdriver.Chrome()
# driver.get("https://layoffs.fyi/")
# time.sleep(5)  # <- this sleep is the whole point of this demo cell
# rows = driver.find_elements(By.CSS_SELECTOR, "table tr")
# print(f"Selenium found {len(rows)} rows after a {5}s forced wait")
# driver.quit()


## 3. Clean — before / after, visibly

In [ ]:
print("BEFORE cleaning:")
display(raw_tracker_df.head(5))

cleaned_df = cl.clean_tracker_dataframe(raw_tracker_df)

print("\nAFTER cleaning:")
display(cleaned_df.head(5))

print(f"\nRows with imputed (not directly reported) headcount: {cleaned_df['_headcount_imputed'].sum()} / {len(cleaned_df)}")


**Look at what just happened to company names.** Run the cell below to see
exactly which raw name variants got merged by the fuzzy dedupe — inspect this
before trusting any company-level aggregation downstream.


In [ ]:
merged = cleaned_df[cleaned_df['company'] != cleaned_df['company_original']][['company_original', 'company']].drop_duplicates()
merged


## 4. EDA — trends by month, sector, size, geography

In [ ]:
eda.plot_monthly_trend(cleaned_df).show()


### 🟡 Checkpoint question (descriptive #1)
> **Is the spike in a given sector a real spike, or a base-rate artifact of
> that sector having more companies tracked?**

*(Left unanswered — look at the chart below before deciding.)*


In [ ]:
eda.plot_by_sector(cleaned_df).show()


In [ ]:
try:
    eda.plot_by_company_size(cleaned_df).show()
except ValueError as e:
    print(f"Skipped: {e}")

try:
    eda.plot_by_geography(cleaned_df).show()
except ValueError as e:
    print(f"Skipped: {e}")


### 🟡 Checkpoint question (descriptive #2)
> **What would a lazy analysis conclude from the monthly trend chart alone —
> and why would it be wrong?**

Hint: look at how much of the recent trend is *imputed* vs *reported*.


In [ ]:
eda.plot_imputed_vs_reported(cleaned_df).show()


## 5. Reason-extraction — stated reasons vs. what the data shows

Companies cite "restructuring." Does the news text support that, or does it
correlate with something happening *before* the announcement (e.g. stock
price)? This step only does simple keyword tagging — deliberately naive, so
its limits are visible rather than hidden behind a black-box classifier.


In [ ]:
if not headlines_df.empty:
    news_context_df = ns.build_news_context_table(headlines_df, max_articles=10)
    display(news_context_df[['title', 'source', 'stated_reasons_found']])
else:
    print("No headlines scraped this run — RSS source may be temporarily empty or blocked.")


### 🟡 Checkpoint question (descriptive #3)
> **Companies cite "restructuring" — does the data support that, or is it PR
> language masking something else (e.g. correlating with stock price *before*
> the announcement)?**


## 6. Forecast — short-horizon prediction by sector, with visible uncertainty

Naive rolling-average baseline **and** ARIMA are shown side by side, on
purpose — never trust a single confident line.


In [ ]:
top_sectors = cleaned_df.groupby('sector')['laid_off'].sum().sort_values(ascending=False).head(6).index.tolist()

sector_dropdown = widgets.Dropdown(options=top_sectors, description='Sector:')
horizon_slider = widgets.IntSlider(value=3, min=1, max=6, step=1, description='Horizon (mo):')

display(sector_dropdown, horizon_slider)


In [ ]:
# Run this cell after picking a sector/horizon above — re-run any time to
# see the forecast (and its uncertainty band) shift live.
sector = sector_dropdown.value
horizon = horizon_slider.value

series = fc.prepare_monthly_series(cleaned_df, sector)
naive_fc = fc.naive_baseline_forecast(series, horizon=horizon)

try:
    arima_fc = fc.arima_forecast(series, horizon=horizon)
except ValueError as e:
    print(f"ARIMA skipped: {e}")
    arima_fc = naive_fc.copy()
    arima_fc['model'] = 'ARIMA_unavailable_fallback_to_naive'

fc.plot_forecast_comparison(series, naive_fc, arima_fc, sector).show()


## 7. Confidence audit — what this forecast rests on, and what's shaky

In [ ]:
audit = fc.confidence_audit(series, sector)
print(f"Sector: {audit['sector']}  |  Months of history: {audit['months_of_history']}\n")
for a in audit['assumptions']:
    print(f"[{a['risk_level']:>6}] {a['assumption']}")
    print(f"         shaky if: {a['shaky_if']}\n")


### 🟡 Checkpoint questions (predictive)
> - **Is this forecast extrapolating a real trend, or just continuing 2 months of noise?**
> - **What's NOT in our data that could break this forecast?** (rate cuts, a single
>   mega-announcement, an AI capex cycle)
> - **If we're right, what should the data look like in 3 months? If we're wrong?**
>   *(forces falsifiability — write down a concrete number before the next class
>   session and check it against what actually happened.)*
> - **Would you personally take a job offer in this sector based on this forecast?**


## 8. Writeup — what's supported, what isn't

Fill this in live as a group, then save as the final deliverable. Suggested skeleton:

1. **What we found**: 2–3 sentences, plain language, no overclaiming.
2. **What a lazy analysis would have concluded, and why it would be wrong.**
3. **What this forecast does NOT account for.**
4. **One falsifiable prediction** to check in 3 months.

This is presented as *one plausible reading of noisy data*, not "the answer."


In [ ]:
writeup = '''
WHAT WE FOUND:


WHAT A LAZY ANALYSIS WOULD HAVE CONCLUDED (AND WHY IT'S WRONG):


WHAT THIS FORECAST DOES NOT ACCOUNT FOR:


ONE FALSIFIABLE PREDICTION TO CHECK IN 3 MONTHS:

'''
with open('data/cleaned/final_writeup.txt', 'w') as f:
    f.write(writeup)
print("Writeup template saved to data/cleaned/final_writeup.txt — fill in and re-save.")


---
## Optional closing demo: live-refreshing Streamlit dashboard

Run this locally (not in Colab) as the 60-second "this is what production
tooling looks like" finale:

```bash
streamlit run streamlit_app.py
```
